In [ ]:
%pip install torch --force-reinstall --index-url https://download.pytorch.org/whl/cu121
%pip install -q peft
%pip install -q datasets
%pip install -q trl

In [1]:
import torch

print(f"Versión de PyTorch: {torch.__version__}")
print(f"¿CUDA disponible?: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print(f"Dispositivos CUDA detectados por driver: {torch.cuda.device_count()}")

Versión de PyTorch: 2.15.0+cu132
¿CUDA disponible?: True


In [2]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

token = os.getenv("TOKEN")

login(token)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",               
    bnb_4bit_compute_dtype=torch.bfloat16, 
    bnb_4bit_use_double_quant=True           
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"                        # Asigna las capas automáticamente a la GPU
)

print("¡Modelo cargado correctamente en VRAM!")

c:\Users\mroscardd\Desktop\Code\IA Keepcoding\proyectos\BOE_Model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1105 14:56:12.670000 15836 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 291/291 [00:06<00:00, 44.07it/s]


¡Modelo cargado correctamente en VRAM!


In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


# 1. Habilitar gradientes y preparar el modelo de 4 bits para entrenamiento
model = prepare_model_for_kbit_training(model)

# 2. Configurar el adaptador LoRA
peft_config = LoraConfig(
    r=16,                           # Rango (rank) de las matrices de adaptación
    lora_alpha=32,                  # Factor de escalado de los gradientes
    target_modules=[                # Capas donde inyectamos los adaptadores
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Aplicar la configuración al modelo
model = get_peft_model(model, peft_config)

# 4. Imprimir parámetros entrenables vs totales
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [5]:
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset


dataset_data = load_dataset(
    "json",
    data_files="./dataset_redaccion_juridica.json",
    split="train"
)


sft_config = SFTConfig(
    output_dir="./resultados_qlora",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=2,
    max_steps=10,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=1,
    optim="paged_adamw_8bit",
    save_strategy="no",
    loss_type="nll",
    max_grad_norm=0.0,
)


# 5. Inicializar SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_data,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss
1,3.154094
2,3.140990
3,2.577176
4,2.051253
5,1.655422
6,1.331135
7,1.105161
8,0.901737
9,0.785612
10,0.733187


TrainOutput(global_step=10, training_loss=1.7435766577720642, metrics={'train_runtime': 18.7044, 'train_samples_per_second': 4.277, 'train_steps_per_second': 0.535, 'total_flos': 810171342618624.0, 'train_loss': 1.7435766577720642, 'epoch': 0.7547169811320755})

In [6]:
model.save_pretrained("./adaptador_qlora")
tokenizer.save_pretrained("./adaptador_qlora")

('./adaptador_qlora\\tokenizer_config.json',
 './adaptador_qlora\\chat_template.jinja',
 './adaptador_qlora\\tokenizer.json')

In [7]:
# 1. Poner el modelo en modo inferencia
model.eval()

# 2. Preparar el prompt
prompt = "### Pregunta: ¿Tengo derecho a una vivienda digna?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 3. Generar respuesta con parámetros de control
with torch.no_grad():
    outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,             # <-- Activa el muestreo para usar temperature y top_p
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id
)

# 4. Decodificar e imprimir
respuesta = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(respuesta)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


### Pregunta: ¿Tengo derecho a una vivienda digna? El derecho a la vivienda es un concepto jurídico que abarca el acceso a una casa, especialmente para las personas con bajos ingresos. Su finalidad principal consiste en proporcionar habitación segura y adecuada, dotada de los servicios básicos necesarios como
